## 1️⃣ Environment Verification

In [ ]:
# Check Python version
import sys
print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")

# Verify Python 3.10+
assert sys.version_info >= (3, 10), "Python 3.10+ required"
print("✅ Python version check passed")

### Import Core Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# System utilities
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

print("✅ Core libraries imported successfully")
print(f"   pandas: {pd.__version__}")
print(f"   numpy: {np.__version__}")
print(f"   plotly: {px.__version__}")

---
## 2️⃣ Project Structure

In [ ]:
# Define project paths
PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
DATA_SAMPLE = DATA_DIR / 'sample'
DATA_RAW = DATA_DIR / 'raw'
DATA_PROCESSED = DATA_DIR / 'processed'

print("📁 Project Structure:")
print(f"   Project root: {PROJECT_ROOT}")
print(f"   Data directory: {DATA_DIR}")
print(f"   Sample data: {DATA_SAMPLE}")
print(f"\n📊 Directory Contents:")

# List directories
for dir_path in [DATA_SAMPLE, DATA_RAW, DATA_PROCESSED]:
    if dir_path.exists():
        files = list(dir_path.glob('*.*'))
        print(f"   {dir_path.name}/: {len(files)} files")
        for f in files[:3]:  # Show first 3
            print(f"      - {f.name}")
    else:
        print(f"   {dir_path.name}/: (not found)")

---
## 3️⃣ Load Sample Data

In [ ]:
# Load sample data
sample_file = DATA_SAMPLE / 'sample_data.csv'

if not sample_file.exists():
    print("❌ Sample data not found!")
    print("   Run: python scripts/generate_sample_data.py")
else:
    df = pd.read_csv(sample_file)
    print(f"✅ Loaded sample data: {sample_file.name}")
    print(f"   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"   Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

### Inspect Data Structure

In [ ]:
# Display first few rows
print("📊 First 5 rows:")
df.head()

In [ ]:
# Data info
print("📋 Dataset Info:")
df.info()

In [ ]:
# Summary statistics
print("📈 Summary Statistics:")
df.describe()

### Check Data Quality

In [ ]:
# Missing values
print("🔍 Missing Values Check:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing': missing,
    'Percentage': missing_pct
}).sort_values('Missing', ascending=False)

print(missing_df[missing_df['Missing'] > 0])

if missing_df['Missing'].sum() == 0:
    print("✅ No missing values!")
else:
    print(f"\n⚠️  {missing_df['Missing'].sum():,} total missing values")

---
## 4️⃣ Basic Data Exploration

In [ ]:
# Key metrics
print("🎯 Key Metrics:")
print(f"   Total devices: {df['device_id'].nunique():,}")
print(f"   Total tests: {df['test_name'].nunique():,}")
print(f"   Lots: {df['lot_id'].nunique()}")
print(f"   Wafers: {df['wafer_id'].nunique()}")

# Pass/Fail breakdown
result_counts = df['result'].value_counts()
print(f"\n📊 Result Distribution:")
for result, count in result_counts.items():
    pct = count / len(df) * 100
    print(f"   {result}: {count:,} ({pct:.1f}%)")

# Overall yield (device level)
device_yield = df.groupby('device_id')['result'].apply(
    lambda x: 'pass' if (x == 'pass').all() else 'fail'
)
yield_rate = (device_yield == 'pass').sum() / len(device_yield) * 100
print(f"\n✨ Overall Yield: {yield_rate:.1f}%")

---
## 5️⃣ Visualization Tests

### Test Distribution Bar Chart

In [ ]:
# Test result distribution
fig = px.bar(
    result_counts.reset_index(),
    x='result',
    y='count',
    title='Test Result Distribution',
    labels={'result': 'Result', 'count': 'Count'},
    color='result',
    color_discrete_map={'pass': 'green', 'fail': 'red'}
)
fig.update_layout(showlegend=False)
fig.show()

print("✅ Plotly bar chart rendered successfully")

### Parametric Test Distribution

In [ ]:
# Filter parametric tests
param_df = df[df['test_type'] == 'parametric'].copy()

if len(param_df) > 0:
    # Histogram of measured values for first parametric test
    first_test = param_df['test_name'].iloc[0]
    test_data = param_df[param_df['test_name'] == first_test]
    
    fig = px.histogram(
        test_data,
        x='measured_value',
        title=f'Distribution of {first_test}',
        labels={'measured_value': 'Measured Value'},
        nbins=30
    )
    
    # Add limit lines
    if 'lower_limit' in test_data.columns:
        ll = test_data['lower_limit'].iloc[0]
        ul = test_data['upper_limit'].iloc[0]
        fig.add_vline(x=ll, line_dash="dash", line_color="red", annotation_text="Lower Limit")
        fig.add_vline(x=ul, line_dash="dash", line_color="red", annotation_text="Upper Limit")
    
    fig.show()
    print("✅ Histogram rendered successfully")
else:
    print("⚠️  No parametric tests found")

### Test Time Analysis

In [ ]:
# Average test time by test
test_time_avg = df.groupby('test_name')['test_time_ms'].mean().sort_values(ascending=False)

fig = px.bar(
    x=test_time_avg.index,
    y=test_time_avg.values,
    title='Average Test Time by Test',
    labels={'x': 'Test Name', 'y': 'Avg Time (ms)'},
    color=test_time_avg.values,
    color_continuous_scale='Viridis'
)
fig.update_layout(showlegend=False)
fig.show()

print("✅ Test time chart rendered successfully")

---
## 6️⃣ Verification Summary

In [ ]:
print("=" * 60)
print("🎉 ENVIRONMENT VERIFICATION COMPLETE")
print("=" * 60)
print("\n✅ Checks Passed:")
print("   [✓] Python 3.10+")
print("   [✓] Core libraries (pandas, numpy, plotly)")
print("   [✓] Project structure")
print("   [✓] Sample data loaded")
print("   [✓] Basic visualizations working")
print("\n🚀 You're ready to proceed!")
print("\n📚 Next Steps:")
print("   1. Open Notebook 01: Data Ingestion")
print("   2. Place your real data files in data/raw/")
print("   3. Explore the analytics modules")
print("=" * 60)

---
## 📝 Notes & Observations

Use this space to record your observations:

**What I learned:**
- 

**Questions:**
- 

**Next actions:**
- 

---

## 🔗 Resources

- **Pandas Documentation:** https://pandas.pydata.org/docs/
- **Plotly Documentation:** https://plotly.com/python/
- **Project README:** ../README.md
- **PRD:** ../PRD.md

---

**Notebook completed! Save your work and proceed to Notebook 01.**